In [39]:
import pandas as pd
from discovery_utils.utils.llm import batch_check

import pandas as pd
import os


data_df = (
    pd.read_csv('afs_open_alex_scan.csv')
    .query("publication_year >= 2000")
    .assign(text = lambda df: df['title'] + ' ' + df['abstract'])
)
len(data_df)

7853

In [82]:
from ast import literal_eval

def extract_author_countries(authorships):
    try:
        # Convert the string representation of the list to an actual list
        authors = literal_eval(authorships)
        countries = []
        for author in authors:
            for institution in author.get('institutions', []):
                if 'country_code' in institution:
                    countries.append(institution['country_code'])
        # remove null values
        countries = [country for country in countries if country is not None]
        return sorted(list(set(countries)))
    except (ValueError, SyntaxError):
        # Handle the case where the string cannot be evaluated
        return []

In [56]:
_data_df = data_df

In [57]:
ids = _data_df.id.tolist()
text = data_df.text.tolist()
test_data = dict(zip(ids, text))

In [58]:
len(test_data)

7852

In [59]:
system_message = (
    "Extract structured information about a measurement or evaluation tool that is used to assess various developmental outcomes "
    "for children under 5 or outcomes for their parents. The tool should be relevant to developmental, behavioural, or well-being outcomes "
    "such as children language development, cognitive development, social-emotional skills, physical and mental health, or "
    "parental outcomes such as confidence, skills, behaviours, knowledge, self-efficacy, and well-being. "
    "The tool could be a questionnaire, structured observation, digital tool or any other form of measurement. "
    "If the input text does not describe such a tool or if input text doesn't useful information (e.g. it's just a list of references)"
    "indicate it is not relevant. When available, provide detailed attributes "
    "such as age suitability, outcome domains, form of measurement, population validated on, and any evidence of effectiveness."
    "Unless requested otherwise, adhere as precisely as possible to the language and text that is used in the provided text document."
    "If the requested information is not described, return N/A. DO NOT make up any false information or false inferences."
)

fields = [
    {"name": "is_relevant", "type": "str", "description": "One-word answer: 'yes' if the text is about a tool for measuring relevant outcomes for children under 5 or their parents, otherwise 'no'."},
    {"name": "is_relevant_reason", "type": "str", "description": "Short explanation (one sentence, 20 words) of why the text is relevant or not."},
    {"name": "tool_type", "type": "str", "description": "Who is the tool used for - one of the following 'Children', 'Parents', 'Both', or 'Other'."},
    {"name": "tool_name", "type": "str", "description": "The name of the measurement or evaluation tool."},
    {"name": "age_range", "type": "str", "description": "The age group the tool is suitable for, as described in the text e.g., '0-5 years', 'infants', 'toddlers'."},
    {"name": "age_range_numerical", "type": "str", "description": "Infer the probable age range in numerical format (e.g., '0-5', '0-2', '3-5')."},
    {"name": "child_outcomes", "type": "list[str]", "description": "List of child-related outcome domains the tool measures, such as 'language development', 'cognitive development', 'social-emotional skills', 'physical health'."},
    {"name": "child_outcomes_category", "type": "list[str]", "description": "Infer if the child-related outcomes are best categorised in one of the following: 'Physical health and development', 'Mental health', 'Social, emotional and behavioral', 'Cognitive development', 'Speech, language and communication', 'N/A'."},
    {"name": "parent_outcomes", "type": "list[str]", "description": "List of parent-related outcomes, such as 'parental confidence', 'knowledge', 'self-efficacy', 'wellbeing'."},
    {"name": "parent_outcomes_category", "type": "list[str]", "description": "Infer if the parent-related outcomes are best categorised in one of the following: 'Parental behaviours and skills', 'Parental beliefs and knowledge', 'Parental well-being and mental health', 'N/A'."},
    {"name": "population_validated", "type": "str", "description": "The population(s) or demographics the tool has been tested or validated with."},
    {"name": "measurement_format", "type": "str", "description": "The format of the tool, e.g., 'questionnaire', 'structured observation', 'parent-reported survey', 'digital app', or others"},
    {"name": "duration_minutes", "type": "int", "description": "Approximate time in minutes it takes to complete the tool, if described."},
    {"name": "evidence", "type": "list[str]", "description": "Explanation of the tools effectiveness or evidence of its impact, if described."},
    {"name": "global_vs_specific", "type": "str", "description": "'global' if the tool measures overall domains (e.g., language development broadly), 'specific' if it targets sub-domains (e.g., gestures, vocalisation), or 'both'."},
    {"name": "ease_of_use", "type": "str", "description": "A qualitative description of the tool’s usability or burden (e.g., easy, moderate, difficult)."},
    {"name": "acceptability", "type": "str", "description": "Acceptability to users (both those completing and administering the tool), if described."},
    {"name": "special_requirements", "type": "str", "description": "Any specific requirements, such as 'needs to be administered twice', 'requires training'."},
    {"name": "geography", "type": "str", "description": "Geographic relevance or validation setting, if specified (e.g., 'UK', 'low-income US settings')."},
    {"name": "country", "type": "str", "description": "Infer country name or names based on the text, if described."},
    {"name": "cost", "type": "str", "description": "Any cost-related information, such as 'free', 'paid license', or specific pricing if stated."},
    {"name": "pros", "type": "list[str]", "description": "List of strengths or advantages of the tool, as stated in the text."},
    {"name": "cons", "type": "list[str]", "description": "List of limitations, drawbacks, or reasons not to use the tool in some cases, as stated in the text."}
]

In [ ]:
processor = batch_check.LLMProcessor(
    model_name="gpt-4.1-mini",
    temperature=0,
    output_path="afs_open_alex_scan_1.jsonl",
    system_message=system_message,
    session_name="afs_open_alex_scan",
    output_fields=fields,
)

processor.run(test_data, batch_size=10, sleep_time=0.5)

2025-04-23 22:16:26,539 - root - INFO - Using OpenAI


<Task pending name='Task-8686' coro=<LLMProcessor.process_text_data() running at /Users/karlis.kanders/Code/discovery_utils/discovery_utils/utils/llm/batch_check.py:120>>

2025-04-23 22:16:26,677 - root - INFO - Processing batch 1/736
2025-04-23 22:16:35,574 - root - INFO - Processing batch 2/736
2025-04-23 22:16:44,496 - root - INFO - Processing batch 3/736
2025-04-23 22:16:50,533 - root - INFO - Processing batch 4/736
2025-04-23 22:16:56,539 - root - INFO - Processing batch 5/736
2025-04-23 22:17:02,588 - root - INFO - Processing batch 6/736
2025-04-23 22:17:11,869 - root - INFO - Processing batch 7/736
2025-04-23 22:17:16,872 - root - INFO - Processing batch 8/736
2025-04-23 22:17:21,554 - root - INFO - Processing batch 9/736
2025-04-23 22:17:27,724 - root - INFO - Processing batch 10/736
2025-04-23 22:17:35,114 - root - INFO - Processing batch 11/736
2025-04-23 22:17:40,531 - root - INFO - Processing batch 12/736
2025-04-23 22:17:45,617 - root - INFO - Processing batch 13/736
2025-04-23 22:17:52,702 - root - INFO - Processing batch 14/736
2025-04-23 22:17:57,431 - root - INFO - Processing batch 15/736
2025-04-23 22:18:03,085 - root - INFO - Processin

In [66]:
data_df.columns

Index(['id', 'doi', 'title', 'display_name', 'relevance_score',
       'publication_year', 'publication_date', 'ids', 'language',
       'primary_location', 'type', 'type_crossref', 'indexed_in',
       'open_access', 'authorships', 'institution_assertions',
       'countries_distinct_count', 'institutions_distinct_count',
       'corresponding_author_ids', 'corresponding_institution_ids', 'apc_list',
       'apc_paid', 'fwci', 'has_fulltext', 'fulltext_origin', 'cited_by_count',
       'citation_normalized_percentile', 'cited_by_percentile_year', 'biblio',
       'is_retracted', 'is_paratext', 'primary_topic', 'topics', 'keywords',
       'concepts', 'mesh', 'locations_count', 'locations', 'best_oa_location',
       'sustainable_development_goals', 'grants', 'datasets', 'versions',
       'referenced_works_count', 'referenced_works', 'related_works',
       'abstract_inverted_index_v3', 'cited_by_api_url', 'counts_by_year',
       'updated_date', 'created_date', 'abstract', 'is_author

In [68]:
data_df.iloc[2]

id                                                 https://openalex.org/W2044379852
doi                                https://doi.org/10.1111/j.1467-789x.2004.00133.x
title                             Obesity in children and young people: a crisis...
display_name                      Obesity in children and young people: a crisis...
relevance_score                                                            4536.706
publication_year                                                               2004
publication_date                                                         2004-04-16
ids                               {'openalex': 'https://openalex.org/W2044379852...
language                                                                         en
primary_location                  {'is_oa': False, 'landing_page_url': 'https://...
type                                                                         review
type_crossref                                                       journal-

In [94]:
cols = [
    'tool_name',
    'doi',
    'is_relevant',
    'is_relevant_reason',    
    'tool_type',            
    'age_range',
    'age_range_numerical',
    'child_outcomes',
    'child_outcomes_category',
    'parent_outcomes',
    'parent_outcomes_category',
    'population_validated',
    'measurement_format',
    'pros',
    'cons',    
    'geography',
    'country',  
    'author_countries', 
    'publication_year', 
    'cited_by_count',          
    'duration_minutes',
    'evidence',
    'global_vs_specific',
    'ease_of_use',
    'acceptability',
    'special_requirements',
    'cost',
    'id',
    'title',
    'text',
]

In [95]:
df_checked = (
    pd.read_json("afs_open_alex_scan_1.jsonl", lines=True)
    .merge(data_df[['id', 'publication_year', 'authorships', 'doi', 'title', 'text', 'is_retracted', 'cited_by_count']], on='id', how='left')
    .assign(
        author_countries = lambda df: df['authorships'].apply(extract_author_countries),
    )
    # convert lists to comma separated strings
    .assign(
        child_outcomes = lambda df: df['child_outcomes'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        child_outcomes_category = lambda df: df['child_outcomes_category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        parent_outcomes = lambda df: df['parent_outcomes'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        parent_outcomes_category = lambda df: df['parent_outcomes_category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        evidence = lambda df: df['evidence'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        pros = lambda df: df['pros'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        cons = lambda df: df['cons'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        author_countries = lambda df: df['author_countries'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    )
    .query("is_retracted == False")
)[cols]
df_checked



,tool_name,doi,is_relevant,is_relevant_reason,tool_type,age_range,age_range_numerical,child_outcomes,child_outcomes_category,parent_outcomes,...,duration_minutes,evidence,global_vs_specific,ease_of_use,acceptability,special_requirements,cost,id,title,text
0,Multiple developmental play assessments includ...,NaN,yes,The text lists multiple play-based assessment ...,Both,"infants, toddlers, preschoolers, young children",0-5,"cognitive development, temperament, social int...","Cognitive development, Social, emotional and b...","parent-child interaction, family interaction, ...",...,-1,These tools are used for assessing development...,both,"varies by tool, generally moderate due to obse...","Not explicitly described, but tools involve pa...",Some tools require trained administrators and ...,N/A,https://openalex.org/W1588659863,Play diagnosis and assessment,Play diagnosis and assessment DEVELOPMENTAL PL...
1,Children's Sleep Habits Questionnaire (CSHQ),https://doi.org/10.1093/sleep/23.8.1d,no,The tool assesses sleep habits in school-aged ...,Children,4-10 years,4-10,"sleep habits, sleep disorders",Physical health and development,,...,0,Adequate internal consistency and test-retest ...,specific,N/A,N/A,N/A,N/A,https://openalex.org/W2146721597,The Children's Sleep Habits Questionnaire (CSH...,The Children's Sleep Habits Questionnaire (CSH...
2,N/A,https://doi.org/10.1111/j.1467-789x.2004.00133.x,no,The text discusses childhood obesity and publi...,N/A,N/A,N/A,,,,...,0,,N/A,N/A,N/A,N/A,N/A,https://openalex.org/W2044379852,Obesity in children and young people: a crisis...,Obesity in children and young people: a crisis...
3,PedsQL™ 4.0 Generic Core Scales,https://doi.org/10.1097/00005650-200108000-00006,yes,The text describes the PedsQL 4.0 tool measuri...,Both,2 to 18 years,2-18,"physical health, emotional health, social heal...","Physical health and development, Mental health...",proxy-report of child's health-related quality...,...,20,Internal consistency reliability acceptable fo...,global,moderate,acceptable to children and parents as per study,N/A,N/A,https://openalex.org/W2323149989,PedsQL™ 4.0: Reliability and Validity of the P...,PedsQL™ 4.0: Reliability and Validity of the P...
4,N/A,https://doi.org/10.1136/bmj.320.7244.1240,no,The text describes BMI cutoffs for child overw...,N/A,N/A,N/A,,,,...,0,,N/A,N/A,N/A,N/A,N/A,https://openalex.org/W2104129218,Establishing a standard definition for child o...,Establishing a standard definition for child o...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7848,Eating in the Absence of Hunger (EAH) question...,https://doi.org/10.1002/eat.22213,no,The tool assesses eating behavior in overweigh...,Children,8-12 years,8-12,"eating behavior, caloric intake",Physical health and development,,...,0,Concurrent validity not supported for EAH ques...,specific,N/A,N/A,N/A,N/A,https://openalex.org/W1941325362,Concurrent and convergent validity of the eati...,Concurrent and convergent validity of the eati...
7849,N/A,https://doi.org/10.1176/appi.ajp.159.9.1541,no,"The tool predicts adolescent violence risk, no...",N/A,N/A,N/A,,,,...,13,Violence Proneness Scale score of 10+ at age 1...,specific,Moderate (13-item scale),N/A,Requires follow-up assessment at age 19,N/A,https://openalex.org/W2155444869,Predicting Adolescent Violence: Impact of Fami...,Predicting Adolescent Violence: Impact of Fami...
7850,Infant Behavior Questionnaire-Revised (IBQ-R),https://doi.org/10.4172/2161-1165.1000167,yes,The text describes the use of the Infant Behav...,Children,6 months old infants,0-1,"infant temperament, orienting and regulation, ...","Social, emotional and behavioral",,...,20,Higher maternal n3:n6 PUFA ratios attenuate ne...,both,moderate,N/A,Requires maternal completion when infant is 6 ...,N/A,https://openalex.org/W1845131196,Effects of prenatal social stress and maternal...,Effects of prenatal social stress and maternal...
7851,Emotional Availability Scales (EAS),https

In [102]:
extra_fields = [
        {"name": "doi", "description": "The DOI of the publication."},
        {"name": "cited_by_count", "description": "The number of times the publication has been cited."},
        {"name": "text", "description": "The text of the publication."},
        {"name": "publication_year", "description": "The year of publication."},
        {"name": "id", "description": "The unique identifier of the publication."},
        {"name": "title", "description": "The title of the publication."},
    ]
all_fields = fields + extra_fields

# Create a table with explanations of all the fields used in the output table (cols columns)
df_explanations = pd.DataFrame.from_records(all_fields).drop(columns=['type'])
# Order it in the same order as the columns in the output table
df_explanations = df_explanations.set_index('name').reindex(cols).reset_index()
df_explanations

,name,description
0,tool_name,The name of the measurement or evaluation tool.
1,doi,The DOI of the publication.
2,is_relevant,One-word answer: 'yes' if the text is about a ...
3,is_relevant_reason,"Short explanation (one sentence, 20 words) of ..."
4,tool_type,Who is the tool used for - one of the followin...
5,age_range,"The age group the tool is suitable for, as des..."
6,age_range_numerical,Infer the probable age range in numerical form...
7,child_outcomes,List of child-related outcome domains the tool...
8,child_outcomes_category,Infer if the child-related outcomes are best c...
9,parent_outcomes,"List of parent-related outcomes, such as 'pare..."


In [104]:
from discovery_utils.utils import (
    google
)
sheet_id = "19BT2NlRcdH_uZG6dFMHtuNHX22WyfmfdpHJOsMkI-UI"

In [106]:
google.upload_data_to_gsheet(sheet_id, {"data": df_checked})
google.format_gsheet(sheet_id, "data", freeze_cols=2)

2025-04-24 10:37:34,958 - root - INFO - Connected to Google Sheet: Tools for measuring children and parents [April 2025]
2025-04-24 10:37:37,262 - root - INFO - Uploading DataFrame to sheet: data
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-04-24 10:39:54,364 - root - INFO - Upload completed successfully.
2025-04-24 10:39:57,519 - root - INFO - Connected to Google Sheet: Tools for measuring children and parents [24-04-2025]


In [ ]:
google.upload_data_to_gsheet(sheet_id, {"info": df_explanations})
google.format_gsheet(sheet_id, "info", freeze_cols=0)